# 01. 레거시 기준선 (개선 전)

이 노트북은 데모를 보는 사람이 **자기 시스템의 개선 전 상태**로 보는 레거시 기준선입니다. `00`과 동일하게, 외부 CLI를 호출하지 않고 노트북 안에서 합성 데이터 생성, 레거시 모델 정의, 시간순 holdout 평가를 모두 직접 수행합니다. 그래서 각 단계를 그대로 열어 분석할 수 있습니다.

대표 레거시 모델은 `Mean`, `Weather`, `ForecastWeather`, `Ldaps`, `SPOT`입니다. 핵심은 예측 시점 입력만 사용하는 `SPOT`이며, 이후 `02`(수동 스킬)·`03`(자동 연구)이 이 `SPOT` 기준선을 넘어서는지 그리고 안전 게이트를 지키는지를 보여줍니다.

- 저장소 루트에서 실행하세요.
- 산출물은 `artifacts/demo/`에 저장되며 `02`가 같은 데이터셋을 입력으로 사용합니다.
- `Weather`는 실제 관측치를 쓰는 사후 분석용 오라클이므로 운영 예측 모델이 아닙니다.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from power_forecasting.data import validate_dataset

SEED = 42
ARTIFACT_DIR = Path("artifacts/demo")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def generate_legacy_demo_data(days=60, plants=3, seed=SEED):
    # 확보할 수 없는 운영 데이터를 대신하는 결정론적 합성 태양광 데이터입니다.
    rng = np.random.default_rng(seed)
    records = []
    for timestamp in pd.date_range("2024-03-01", periods=days * 24, freq="h"):
        daylight = max(0.0, math.sin(math.pi * (timestamp.hour - 6) / 12))
        seasonal = 0.80 + 0.18 * math.cos(2 * math.pi * (timestamp.dayofyear - 172) / 365)
        for plant_index in range(plants):
            capacity_mw = 42.0 + plant_index * 13.0
            actual_cloud_cover = float(np.clip(0.40 + 0.12 * math.sin(2 * math.pi * (timestamp.hour + plant_index) / 24) + rng.normal(0, 0.07), 0, 1))
            actual_temperature = 20 + 7 * seasonal + 7 * math.sin(2 * math.pi * (timestamp.hour - 8) / 24) + rng.normal(0, 1.2)
            actual_wind_speed = float(np.clip(4 + rng.normal(0, 0.8), 0, None))
            clear_sky_irradiance = 990 * daylight * seasonal
            actual_irradiance = float(np.clip(clear_sky_irradiance * (1 - 0.72 * actual_cloud_cover) + rng.normal(0, 18 * daylight), 0, None))
            forecast_cloud_cover = float(np.clip(actual_cloud_cover + rng.normal(0.02, 0.11), 0, 1))
            forecast_temperature = float(actual_temperature + rng.normal(0.2, 2.0))
            forecast_wind_speed = float(np.clip(actual_wind_speed + rng.normal(0, 1.1), 0, None))
            forecast_irradiance = float(np.clip(clear_sky_irradiance * (1 - 0.67 * forecast_cloud_cover) + rng.normal(0, 45 * daylight), 0, None))
            ldaps_cloud_cover = float(np.clip(actual_cloud_cover + rng.normal(0, 0.07), 0, 1))
            ldaps_temperature = float(actual_temperature + rng.normal(0, 1.1))
            ldaps_irradiance = float(np.clip(actual_irradiance + rng.normal(0, 28 * daylight), 0, None))
            ldaps_humidity = float(np.clip(58 + 28 * actual_cloud_cover - 0.45 * (actual_temperature - 25) + rng.normal(0, 4), 5, 100))
            generation_mw = float(np.clip(capacity_mw * (actual_irradiance / 1000) * (0.90 + plant_index * 0.025) * (1 - 0.0045 * max(0, actual_temperature - 25)), 0, capacity_mw))
            records.append({"plant_id": f"plant_{plant_index + 1:02d}", "timestamp": timestamp, "capacity_mw": capacity_mw, "latitude": 33.0 + plant_index * 2.7, "longitude": 126.0 + plant_index * 1.9, "actual_irradiance": actual_irradiance, "actual_temperature": actual_temperature, "actual_cloud_cover": actual_cloud_cover, "actual_wind_speed": actual_wind_speed, "forecast_irradiance": forecast_irradiance, "forecast_temperature": forecast_temperature, "forecast_cloud_cover": forecast_cloud_cover, "forecast_wind_speed": forecast_wind_speed, "ldaps_irradiance": ldaps_irradiance, "ldaps_temperature": ldaps_temperature, "ldaps_cloud_cover": ldaps_cloud_cover, "ldaps_humidity": ldaps_humidity, "generation_mw": generation_mw})
    return pd.DataFrame(records).sort_values(["timestamp", "plant_id"]).reset_index(drop=True)


def chronological_holdout(frame, train_fraction=0.8):
    timestamps = frame["timestamp"].drop_duplicates().sort_values().to_numpy()
    cutoff = timestamps[int(len(timestamps) * train_fraction)]
    return frame.loc[frame["timestamp"] < cutoff].copy(), frame.loc[frame["timestamp"] >= cutoff].copy()


def metric_values(frame):
    error = frame["prediction_mw"] - frame["generation_mw"]
    return {"MAE": float(error.abs().mean()), "RMSE": float(np.sqrt((error ** 2).mean())), "NMAE": float(error.abs().sum() / frame["capacity_mw"].sum())}


def ridge_pipeline():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))


def predict_mean(train, holdout):
    hourly = train.assign(hour=train["timestamp"].dt.hour).groupby(["plant_id", "hour"])["generation_mw"].mean()
    fallback = train.groupby("plant_id")["generation_mw"].mean()
    keys = pd.MultiIndex.from_arrays([holdout["plant_id"], holdout["timestamp"].dt.hour])
    values = hourly.reindex(keys).to_numpy(dtype=float)
    return np.where(np.isnan(values), holdout["plant_id"].map(fallback).to_numpy(), values)


def evaluate(name, availability, features=None, factory=None):
    if name == "Mean":
        prediction = predict_mean(train, holdout)
    else:
        estimator = factory().fit(train[features], train["generation_mw"])
        prediction = estimator.predict(holdout[features])
    result = holdout[["plant_id", "timestamp", "generation_mw", "capacity_mw"]].copy()
    result["prediction_mw"] = np.clip(prediction, 0, result["capacity_mw"])
    result["model"] = name
    result["availability"] = availability
    return result


dataset = generate_legacy_demo_data()
validate_dataset(dataset)
train, holdout = chronological_holdout(dataset)
WEATHER_FEATURES = ["actual_irradiance", "actual_temperature", "actual_cloud_cover", "actual_wind_speed", "capacity_mw"]
FORECAST_WEATHER_FEATURES = ["forecast_irradiance", "forecast_temperature", "forecast_cloud_cover", "forecast_wind_speed", "capacity_mw"]
LDAPS_FEATURES = ["ldaps_irradiance", "ldaps_temperature", "ldaps_cloud_cover", "ldaps_humidity", "capacity_mw"]
SPOT_FEATURES = FORECAST_WEATHER_FEATURES + ["latitude", "longitude"]
models = {
    "Mean": evaluate("Mean", "historical"),
    "Weather": evaluate("Weather", "actual (oracle)", WEATHER_FEATURES, ridge_pipeline),
    "ForecastWeather": evaluate("ForecastWeather", "forecast", FORECAST_WEATHER_FEATURES, ridge_pipeline),
    "Ldaps": evaluate("Ldaps", "forecast", LDAPS_FEATURES, ridge_pipeline),
    "SPOT": evaluate("SPOT", "forecast (AIDM baseline)", SPOT_FEATURES, lambda: make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingRegressor(random_state=SEED, max_iter=100))),
}
model_metrics = pd.DataFrame([{"model": name, "availability": frame["availability"].iat[0], **metric_values(frame)} for name, frame in models.items()]).sort_values("NMAE").reset_index(drop=True)
model_metrics

In [ ]:
# 이 데이터셋과 SPOT 예측을 02(수동)·03(자동)의 입력 증적으로 남깁니다.
dataset.to_csv(ARTIFACT_DIR / "dataset.csv", index=False)
legacy_predictions = models["SPOT"][["plant_id", "timestamp", "prediction_mw"]]
assert not legacy_predictions.duplicated(["plant_id", "timestamp"]).any()
legacy_predictions.to_csv(ARTIFACT_DIR / "legacy_predictions.csv", index=False)

spot_nmae = model_metrics.loc[model_metrics["model"] == "SPOT", "NMAE"].iat[0]
print(f"개선 전 SPOT 기준선 NMAE: {spot_nmae:.6f}")
print(f"데이터셋: {ARTIFACT_DIR / 'dataset.csv'}")
print(f"SPOT 예측: {ARTIFACT_DIR / 'legacy_predictions.csv'}")
print("이 데이터셋과 SPOT 기준선이 02(수동)와 03(자동)의 출발점입니다.")

## 다음 단계

여기까지가 개선 전 레거시입니다. 같은 데이터셋과 `SPOT` 기준선을 놓고 두 가지 개선 경로를 보여줍니다.

- `02_manual_skill_path.ipynb`: 사람이 통제하는 기본 경로 — `legacy-intake -> AIDM experiment -> AIDD promotion -> human review`.
- `03_auto_research_path.ipynb`: 선택적 자동 Stage 1 연구 경로 — `diagnosis -> bounded proposal -> AIDM -> evidence verification -> human review`.

두 경로 모두 승격 게이트를 우회하지 않으며, 실제 배포는 사람 검토 이후에만 이뤄집니다.